###Rag pipeline - Data injestion to vector DB

In [1]:
import os 
import re
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Gaurang\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf inside the directory
def process_all_pdfs(pdf_directory):
    """Process all the pdf in the directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    ###Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to meatadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" ✅Loaded {len(documents)} pages")

        except Exception as e:
            print(f"error :{e}")
    print(f"\nTotal documents loaded : {len(all_documents)}")
    return all_documents

# Process all the PDF's in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 1 PDF files to process

Processing : Firewallp.pdf
 ✅Loaded 12 pages

Total documents loaded : 12


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2024-05-18T13:12:32+05:30', 'source': '..\\data\\pdf_files\\Firewallp.pdf', 'file_path': '..\\data\\pdf_files\\Firewallp.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': 'flaviag', 'subject': '', 'keywords': '', 'moddate': '2024-05-18T13:12:32+05:30', 'trapped': '', 'modDate': "D:20240518131232+05'30'", 'creationDate': "D:20240518131232+05'30'", 'page': 0, 'source_file': 'Firewallp.pdf', 'file_type': 'pdf'}, page_content='. \n \n \n \nFIREWALL CHARACTERISTICS \n \nThe characteristics of firewall are \n1. Service Control \n2. Direction Control \n3. User Control \n4. Behaviour Control \n● \nService Control: Determines the types of Internet services that can be accessed by the \nnetwork user. The inbound or outbound traffic may be filtered based on the basis of IP \naddress and TCP port number. It can be implemented by proxy software or host on the \nserver softw

In [4]:
"""import re

def is_bad_chunk(text):
    words = text.split()

    # 1️⃣ Too many uppercase fragments (like INFO RMA TION)
    uppercase_words = [w for w in words if w.isupper() and len(w) <= 5]
    if len(uppercase_words) > 5:
        return True

    # 2️⃣ Detect broken word pattern (e.g., "Intro ductio n")
    broken_pattern = re.findall(r'\b\w{2,5}\s+\w{2,5}\s+\w{1,4}\b', text)
    if len(broken_pattern) > 2:
        return True

    # 3️⃣ Too many line breaks (vertical text)
    if text.count("\n") > 8:
        return True

    # 4️⃣ Too many very short words
    short_words = [w for w in words if len(w) <= 2]
    if len(short_words) / max(len(words), 1) > 0.3:
        return True

    return False"""

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Gaurang\AppData\Local\Temp\ipykernel_9880\1689116559.py:1: SyntaxWarning: invalid escape sequence '\w'
  """import re


'import re\n\ndef is_bad_chunk(text):\n    words = text.split()\n\n    # 1️⃣ Too many uppercase fragments (like INFO RMA TION)\n    uppercase_words = [w for w in words if w.isupper() and len(w) <= 5]\n    if len(uppercase_words) > 5:\n        return True\n\n    # 2️⃣ Detect broken word pattern (e.g., "Intro ductio n")\n    broken_pattern = re.findall(r\'\x08\\w{2,5}\\s+\\w{2,5}\\s+\\w{1,4}\x08\', text)\n    if len(broken_pattern) > 2:\n        return True\n\n    # 3️⃣ Too many line breaks (vertical text)\n    if text.count("\n") > 8:\n        return True\n\n    # 4️⃣ Too many very short words\n    short_words = [w for w in words if len(w) <= 2]\n    if len(short_words) / max(len(words), 1) > 0.3:\n        return True\n\n    return False'

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} docs into {len(split_docs)} chunks")

    return split_docs


In [6]:
###Get all the chunks
chunks = split_documents(all_pdf_documents)
chunks


Split 12 docs into 23 chunks


[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2024-05-18T13:12:32+05:30', 'source': '..\\data\\pdf_files\\Firewallp.pdf', 'file_path': '..\\data\\pdf_files\\Firewallp.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': 'flaviag', 'subject': '', 'keywords': '', 'moddate': '2024-05-18T13:12:32+05:30', 'trapped': '', 'modDate': "D:20240518131232+05'30'", 'creationDate': "D:20240518131232+05'30'", 'page': 0, 'source_file': 'Firewallp.pdf', 'file_type': 'pdf'}, page_content='. \n \n \n \nFIREWALL CHARACTERISTICS \n \nThe characteristics of firewall are \n1. Service Control \n2. Direction Control \n3. User Control \n4. Behaviour Control \n● \nService Control: Determines the types of Internet services that can be accessed by the \nnetwork user. The inbound or outbound traffic may be filtered based on the basis of IP \naddress and TCP port number. It can be implemented by proxy software or host on the \nserver softw

###Embedding and Vector Store DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer #Embedding model available inside this
import chromadb
from chromadb.config import Settings
import uuid #every record stored inside vector db will have somme unique id so we will use this to generate that
from typing import List,Dict,Any,Tuple 
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
from typing import List
import numpy as np

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize Embeddings manager
        Args:
            model name: HuggingFace Model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model() #imp - load this model

    #using _ before name to create an protected class that can be acceced within only this file
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedings dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name: {e}}")
            raise
    
    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("ModuleNotLoaded")
        
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated Embeddings with shape : {embeddings.shape}")
        return embeddings

## initialize the embbeding manager
embdding_manager = EmbeddingManager()
embdding_manager

Loading embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2716.09it/s]


Model loaded sucessfully. Embedings dimension: 384


C:\Users\Gaurang\AppData\Local\Temp\ipykernel_9880\663904986.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded sucessfully. Embedings dimension: {self.model.get_sentence_embedding_dimension()}")


In [9]:
import os
class VectorStore:
    """Manage document embeddings in a chromaDB vectore store"""

    def __init__(self , collection_name: str="pdf_documents",persist_directory:str="./data/vectore_store"):
        """Initialize the vectore store
        
        Args:
            collection_name = name of the ChromaDB collection
            persist_directory:directory to persist the vector store
        
        """

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initalize_store()

    def reset_collection(self):
        """Delete and recreate the collection to avoid duplicate chunks across notebook reruns."""
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            pass

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embedding for RAG"}
        )
        print(f"Collection {self.collection_name} reset. Current count: {self.collection.count()}")

    def _initalize_store(self):
        try:
            #Create persistent ChromaDB collection
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embedding for RAG"}
            )
            print(f"Vector store initialized. Collection {self.collection_name}")
            print(f"Existing document in collection {self.collection.count()}")

        except Exception as e:
            print(f"Error in initializing the store",{e})
            raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        Add document and their embeddings to the vector store

        Args:
            Documents:List of the LangChain documents
            embeddings: corresponding embeddings for the documents

        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documnets must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for chromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i , (doc,embedding) in enumerate(zip(documents,embeddings)):
            #Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #document content
            documents_text.append(doc.page_content)

            #Embeddings
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documnets to the vectoe store")
            print(f"Total documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error in adding documnets in the vector store : {e}")
            raise

vectorstore=VectorStore()
vectorstore.reset_collection()
vectorstore

Vector store initialized. Collection pdf_documents
Existing document in collection 23
Collection pdf_documents reset. Current count: 0


In [10]:
###Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

###Generate the embeddings
embeddings=embdding_manager.generate_embeddings(texts)

###Store in the vector database
vectorstore.add_documents(chunks,embeddings)

# See actual stored chunks
results = vectorstore.collection.get(limit=5)
for i, doc in enumerate(results['documents'][:3]):
    print(f"Chunk {i}:\n{doc}\n---")

Generating embeddings for 23 texts


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.33s/it]

Generated Embeddings with shape : (23, 384)
Adding 23 documents to vector store...
Successfully added 23 documnets to the vectoe store
Total documents in the collection: 23
Chunk 0:
. 
 
 
 
FIREWALL CHARACTERISTICS 
 
The characteristics of firewall are 
1. Service Control 
2. Direction Control 
3. User Control 
4. Behaviour Control 
● 
Service Control: Determines the types of Internet services that can be accessed by the 
network user. The inbound or outbound traffic may be filtered based on the basis of IP 
address and TCP port number. It can be implemented by proxy software or host on the 
server software. 
● 
Direction control: Determines the direction such as inbound or outbound in which 
particular service requests are allowed to flow through the Firewall. 
● 
User control: Controls access to a service according to the user. Each user will have a 
ACLs indicates their level of access. Based on ACL the user traffic may allowed or 
denied. This feature is typically applied to user

###Retrieval pipeline from vector store

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
            Initialize the retriever

            Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def _keyword_overlap_score(self, query: str, text: str) -> float:
        query_terms = {word for word in re.findall(r"[a-zA-Z]+", query.lower()) if len(word) > 2}
        text_terms = set(re.findall(r"[a-zA-Z]+", text.lower()))
        if not query_terms:
            return 0.0
        return len(query_terms & text_terms) / len(query_terms)

    def _is_toc_like_chunk(self, text: str) -> bool:
        numbered_sections = len(re.findall(r"\b\d+\.\d+\b", text))
        short_lines = sum(1 for line in text.splitlines() if 0 < len(line.strip()) <= 30)
        return numbered_sections >= 4 or short_lines >= 12 or "unit structure" in text.lower()

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query

            Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

            Returns:
            List of dictionaries containing retrieved documents and metadata

            print(f"Retrieving documents for query: '(quepy}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        """
        print(f"Retriving documents for query : '{query}'")
        print(f"Top k : {top_k},Score threshold: {score_threshold}")

        #Generate query embeddings
        query_embeddings= self.embedding_manager.generate_embeddings([query])[0]

        #Search in vectore store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=max(top_k * 4, 10)
            )

            #Process the results
            retrieved_docs =[]
            seen_contents = set()

            if results['documents'] and results['documents'][0]:
                documents  = results['documents'][0]
                metadatas  = results['metadatas'][0]
                distances  = results['distances'][0]
                ids= results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    distance = float(distance)   
                    similarity_score = 1 - distance
                    keyword_score = self._keyword_overlap_score(query, document)
                    penalty = 0.15 if self._is_toc_like_chunk(document) else 0.0
                    combined_score = similarity_score + (0.35 * keyword_score) - penalty

                    normalized_content = " ".join(document.split())
                    if normalized_content in seen_contents:
                        continue

                    if combined_score >= score_threshold:
                        seen_contents.add(normalized_content)
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'keyword_score': keyword_score,
                            'combined_score': combined_score,
                            'distance': distance,
                            'rank':i+1
                        })
                retrieved_docs = sorted(retrieved_docs, key=lambda x: x['combined_score'], reverse=True)[:top_k]
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval : {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embdding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("TYPES OF FIREWALLS")

Retriving documents for query : 'TYPES OF FIREWALLS'
Top k : 5,Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.99it/s]

Generated Embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_6dd7b247_1',
  'content': 'outbound traffic. It may also be applied to incoming traffic from external users, but it \nneeds authentication technique \n\uf0b7 Behavior control: It makes use of statistical data to control the traffic. Controls how \nparticular services are used (e.g. filter e-mail to eliminate spam), or it may enable external access to \nonly a portion of the information on a local Web server. \n \nTYPES OF FIREWALLS \n \nThere are three types of firewall namely, \n1) Packet Filtering \n2) Application Gateways \n3) Circuit-level Gateways \n1) Packet Filtering: \nFirewalls filter each packet that attempt to enter or leave a private network \nand either accept or reject them depending on the predefined set of filter \nrules which is based on based on pattern-matching. Figure 5.8 shows the \nplacement of packet filtering firewall which is between the public and \nprivate network.',
  'metadata': {'producer': 'Microsoft® Word 2013',
   'trapped': '',
   'page': 

Integration Vectordb context pipeline with LLM output

In [14]:
import langchain_groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set in the environment.")

llm = ChatGroq(api_key=groq_api_key, model_name=groq_model, temperature=0.1, max_tokens=1024)

# Cell 3 — RAG function
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Build prompt
    prompt = f"""Use the following context to answer the question.
        If the context contains a list of principles, definitions, or bullet points, summarize them clearly.
        Do not answer with only a heading or section title.
        If the answer is present, provide the actual points from the context.

        Context:{context}

        Question: {query}

        Answer:"""

    # Call Groq LLM
    #response = llm.invoke([HumanMessage(content=prompt)])
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [15]:
# Step 1 — Check what chunks were actually retrieved
results = rag_retriever.retrieve("The characteristics of firewall are", top_k=5)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc['content'][:300])

Retriving documents for query : 'The characteristics of firewall are'
Top k : 5,Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.62it/s]

Generated Embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)

--- Chunk 1 ---
. 
 
 
 
FIREWALL CHARACTERISTICS 
 
The characteristics of firewall are 
1. Service Control 
2. Direction Control 
3. User Control 
4. Behaviour Control 
● 
Service Control: Determines the types of Internet services that can be accessed by the 
network user. The inbound or outbound traffic may be f

--- Chunk 2 ---
outbound traffic. It may also be applied to incoming traffic from external users, but it 
needs authentication technique 
 Behavior control: It makes use of statistical data to control the traffic. Controls how 
particular services are used (e.g. filter e-mail to eliminate spam), or it may enable e

--- Chunk 3 ---
tables, the packets are forwarded; otherwise, the packets are dropped or 
rejected. 
 
Figure 10.3 Stateful inspection 
Stateless Packet Filter: 
Stateless packet filtering firewalls are perhaps the oldest and most 
established firewall option. While they're less 

In [16]:
# Step 2 — Then get the answer separately
answer = rag_simple("The characteristics of firewall are", rag_retriever, llm)
print("\n--- FINAL ANSWER ---")
print(answer)

Retriving documents for query : 'The characteristics of firewall are'
Top k : 3,Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.97it/s]

Generated Embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)



--- FINAL ANSWER ---
The characteristics of a firewall are:

1. Service Control: Determines the types of Internet services that can be accessed by the network user. The inbound or outbound traffic may be filtered based on the basis of IP address and TCP port number. It can be implemented by proxy software or host on the server software.

2. Direction Control: Determines the direction such as inbound or outbound in which particular service requests are allowed to flow through the Firewall.

3. User Control: Controls access to a service according to the user. Each user will have ACLs (Access Control Lists) that indicate their level of access. Based on ACL, user traffic may be allowed or denied.

4. Behaviour Control: It makes use of statistical data to control the traffic. Controls how particular services are used (e.g. filter e-mail to eliminate spam), or it may enable external access to only a portion of the information on a local Web server.


Enhance Rag Pipeline

In [17]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response =  llm.invoke([prompt.format(context=context,query=query)])

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Screened Subnet Firewall?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retriving documents for query : 'What is Screened Subnet Firewall?'
Top k : 3,Score threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.23it/s]

Generated Embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)


Answer: A Screened Subnet Firewall is a configuration that offers the highest security among possible firewall configurations. It uses two packet filtering routers, one between the bastion host and the internet and one between the bastion host and the internal network, to create an isolated sub-network that blocks traffic across the subnet.
Sources: [{'source': 'Firewallp.pdf', 'page': 10, 'score': 0.42862534523010254, 'preview': 'Firewall \n \n \n \n \n \n \n \n \n \n \n \nFigure 10.8 Dual-homed bastion \nScreened Subnet Firewall:- \nIt offers the highest security among the possible firewall configurations. It \nis an improvement over the previous scheme of screened host firewall, dual-\nhomed bastion. Two packet filtering routers are used,...'}, {'source': 'Firewallp.pdf', 'page': 10, 'score': 0.3363901972770691, 'preview': 'advantages: \n\uf0b7 \nThere are now three levels of defense to prevent intruders. \n\uf0b7 \nThe outside router advertises only the existence of the screened su

In [18]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k)

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            # Combine document content
            context = "\n\n".join([doc["content"] for doc in results])

            # Extract metadata for citations
            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file", doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc.get("similarity_score", 0),
                }
                for doc in results
            ]

            # Build prompt
            prompt = f"""
You are an AI assistant. Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

            # Generate answer
            if stream:
                answer = ""
                for chunk in self.llm.stream(prompt):
                    text = chunk.content if hasattr(chunk, "content") else str(chunk)
                    print(text, end="", flush=True)
                    answer += text
                print()
            else:
                answer = self.llm.generate(prompt)

        # Optional summarization
        summary = None
        if summarize and context:
            summary_prompt = f"Summarize the following context:\n{context}"
            summary = self.llm.invoke(summary_prompt).content
        # Store history
        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "timestamp": time.time(),
            }
        )

        return {
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary,
            "history": self.history,   # add this
            "history_length": len(self.history),
        }
    
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Firewall?", top_k=3, min_score=0.3, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary' ])
print("History:", result['history'][-1])

Retriving documents for query : 'What is Firewall?'
Top k : 3,Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]

Generated Embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)
A firewall is a network security system that monitors and controls incoming and outgoing network traffic based on predetermined security rules. It acts as a barrier between a trusted network

 (such as a private network) and an untrusted network (such as the internet), preventing unauthorized access to the private network and protecting it from various types of cyber threats.

Firewalls can be implemented in various ways, including hardware, software, or a combination of both. They can be configured to control traffic in both directions, allowing or blocking specific types of traffic based on factors such as IP address, port number, and protocol.

Firewalls can also be used to control user access to specific services, such as email or web browsing, and to enforce security policies, such as blocking malicious traffic or restricting access to certain websites.

There are three main types of firewalls:

1. Packet Filtering: This type of firewall examines each packet of data and allows or blocks it based on predetermined rules.
2. Application Gateways: This type of firewall examines the data at the application layer and allows or blocks it based on the type of application being